# Brand and Category Historical Ratios

The code chunks below were run on a local machine but accessed data from a local image of Google Drive.

## 1. Install and configure

Metadata parquet must contain `wid`, `brand_id`, and `cate_id_4`.

In [ ]:
from collections import Counter, OrderedDict
from pathlib import Path
from numbers import Integral
from urllib.parse import quote
import hashlib
import json
import os
import sqlite3
import tempfile
import time
import uuid

PROJECT = Path("/content/drive/MyDrive/Grp8Submission/pipeline_data")
INPUT_PATHS = {
    "train": DATA_DIR / "train_sampled.parquet",
    "val": DATA_DIR / "val.parquet",
    "test": DATA_DIR / "test.parquet",
}

PRODUCT_METADATA_PATH = DATA_DIR / "product_metadata_clean.parquet"  # file or shard directory
METADATA_COLUMNS = ("wid", "brand_id", "cate_id_4")
HISTORY_COLUMN = "history_wid_list"
CANDIDATE_COLUMN = "candidate_wid_list"

OUTPUT_DIR = DATA_DIR / "model2_inputs/Dom"
OUTPUT_LAYOUT = "slate"          # "candidate" gives scalar features per candidate
BATCH_SIZE = 128                 # source slates per batch; reduce for very long histories
METADATA_BATCH_SIZE = 100_000    # read only the three metadata columns
LOOKUP_CACHE_SIZE = 200_000      # bounded product-ID cache; does not store labels/history
REBUILD_METADATA_INDEX = True
OVERWRITE_OUTPUTS = True

# Treat only null attributes as unknown by default. Add dataset-documented
# sentinel IDs here if needed, for example {0, -1}; never assume 0 is missing.
UNKNOWN_BRAND_IDS = set()
UNKNOWN_CATEGORY_IDS = set()

for name, path in {**INPUT_PATHS, "metadata": PRODUCT_METADATA_PATH}.items():
    print(f"{name}: {'Found' if path.exists() else 'MISSING'} — {path}")


train: Found — /Users/domi_bryan/Library/CloudStorage/GoogleDrive-domi@comp.nus.edu.sg/.shortcut-targets-by-id/1-dwM_44UXnFberNRNLzd2YLPRzRuRUUw/BT4222 Project Deliverables/data/JDSearch/train_sampled.parquet
val: Found — /Users/domi_bryan/Library/CloudStorage/GoogleDrive-domi@comp.nus.edu.sg/.shortcut-targets-by-id/1-dwM_44UXnFberNRNLzd2YLPRzRuRUUw/BT4222 Project Deliverables/data/JDSearch/val.parquet
test: Found — /Users/domi_bryan/Library/CloudStorage/GoogleDrive-domi@comp.nus.edu.sg/.shortcut-targets-by-id/1-dwM_44UXnFberNRNLzd2YLPRzRuRUUw/BT4222 Project Deliverables/data/JDSearch/test.parquet
metadata: Found — /Users/domi_bryan/Library/CloudStorage/GoogleDrive-domi@comp.nus.edu.sg/.shortcut-targets-by-id/1-dwM_44UXnFberNRNLzd2YLPRzRuRUUw/BT4222 Project Deliverables/data/JDSearch/product_metadata_clean.parquet


## 2. Definitions and edge cases

Let $H=(h_1, \ldots, h_n)$ be a row's full history, $b(w)$ its product's
brand, $g(w)$ its level-4 category, and $c$ a candidate product.

$$
\operatorname{brand\_affinity\_ratio}(H,c)
  =\frac{\sum_{j=1}^{n}\mathbf{1}[b(h_j)=b(c)\text{ and both are known}]}{n}
$$

$$
\operatorname{cate\_affinity\_ratio}(H,c)
  =\frac{\sum_{j=1}^{n}\mathbf{1}[g(h_j)=g(c)\text{ and both are known}]}{n}
$$

| Feature | Rule | Slate output type | Candidate output type |
|---|---|---|---|
| `hist_dominant_brand` | Most frequent known history brand | nullable int64 | nullable int64 |
| `brand_affinity_ratio` | Matching-brand interaction count / full history length | list<float32> | float32 |
| `cate_affinity_ratio` | Matching-L4 interaction count / full history length | list<float32> | float32 |

- Each occurrence counts. Repeated products are not deduplicated; all supplied
  interaction types contribute equally. No recency weighting or truncation.
- Dominant-brand ties use the **smallest numeric brand ID**. A history with
  no known brand has a null dominant brand.
- Missing history metadata and null history entries contribute no matches
  but **remain in the denominator**. Unknown values never match one another.
- Empty/null histories and unknown candidate attributes have affinity `0.0`.
  Brand and category missingness are handled independently.
- A null candidate entry is retained at its position with zero affinities.
  Empty/null candidate lists produce empty affinity lists and zero flat rows.
- Duplicate metadata product IDs are rejected, including identical duplicates.
  Resolve them to one authoritative row per `wid` before running.
- IDs remain signed int64, with no rounding or conversion through floating point.
  `hist_dominant_brand` is a **categorical identifier**, not an ordinal quantity.

**Example:** history brands `[10, 20, 10, 10, unknown, unknown]` give dominant
brand `10`. Brand-10 and brand-20 candidates receive `3/6` and `1/6`.


In [2]:
FEATURE_NAMES = ("hist_dominant_brand", "brand_affinity_ratio", "cate_affinity_ratio")
INT64_MIN, INT64_MAX = -(2**63), 2**63 - 1


def integer_id(value, *, allow_none=True):
    """Never round an ID or silently convert floats/strings to integers."""
    if value is None and allow_none:
        return None
    if isinstance(value, bool) or not isinstance(value, Integral):
        raise TypeError(f"Expected an integer product/attribute ID, got {value!r}")
    value = int(value)
    if not INT64_MIN <= value <= INT64_MAX:
        raise OverflowError(f"ID is outside signed int64: {value}")
    return value


def normalise_sequence(values):
    if values is None:
        return []
    if not isinstance(values, (list, tuple)):
        raise TypeError("Expected a Parquet list column, not a serialized string.")
    return [integer_id(v) for v in values]


def engineer_slate(history, candidates, products):
    """Inputs are normalized lists; products maps wid -> (brand, L4) or None.

    Repeated history interactions count repeatedly. Every supplied history
    position, including a missing product/attribute, remains in the denominator.
    """
    brands, categories = Counter(), Counter()
    stats = {
        "history_events": len(history),
        "history_missing_product": 0,
        "history_missing_brand": 0,
        "history_missing_category": 0,
        "candidate_events": len(candidates),
        "candidate_missing_product": 0,
        "candidate_missing_brand": 0,
        "candidate_missing_category": 0,
        "empty_histories": int(not history),
    }
    for wid in history:
        attrs = products.get(wid) if wid is not None else None
        brand, category = attrs if attrs is not None else (None, None)
        stats["history_missing_product"] += int(attrs is None)
        stats["history_missing_brand"] += int(brand is None)
        stats["history_missing_category"] += int(category is None)
        if brand is not None:
            brands[brand] += 1
        if category is not None:
            categories[category] += 1

    # Numerically smallest known brand ID wins a frequency tie.
    dominant = min(brands, key=lambda b: (-brands[b], b)) if brands else None
    stats["histories_without_known_brand"] = int(dominant is None)
    brand_ratios, category_ratios = [], []
    denominator = len(history)
    for wid in candidates:
        attrs = products.get(wid) if wid is not None else None
        brand, category = attrs if attrs is not None else (None, None)
        stats["candidate_missing_product"] += int(attrs is None)
        stats["candidate_missing_brand"] += int(brand is None)
        stats["candidate_missing_category"] += int(category is None)
        brand_ratios.append(
            brands.get(brand, 0) / denominator
            if denominator and brand is not None else 0.0
        )
        category_ratios.append(
            categories.get(category, 0) / denominator
            if denominator and category is not None else 0.0
        )
    return dominant, brand_ratios, category_ratios, stats


class ProductLookup:
    """Disk-backed lookup with a bounded cache shared across all three splits."""

    def __init__(self, path, cache_size=200_000):
        if cache_size < 0:
            raise ValueError("cache_size must be nonnegative")
        uri = "file:" + quote(str(Path(path).resolve())) + "?mode=ro"
        self.connection = sqlite3.connect(uri, uri=True)
        self.connection.execute("PRAGMA cache_size=-65536")
        self.cache = OrderedDict()
        self.cache_size = cache_size

    def resolve(self, ids):
        result, missing = {}, []
        for wid in set(ids):
            if wid is None:
                continue
            if wid in self.cache:
                result[wid] = self.cache[wid]
                self.cache.move_to_end(wid)
            else:
                missing.append(wid)
        # 900 parameters also works on SQLite builds with a 999-variable limit.
        for offset in range(0, len(missing), 900):
            chunk = missing[offset:offset + 900]
            placeholders = ",".join("?" for _ in chunk)
            sql = (
                "SELECT wid, brand_id, cate_id_4 FROM products "
                f"WHERE wid IN ({placeholders})"
            )
            found = {w: (b, c) for w, b, c in self.connection.execute(sql, chunk)}
            for wid in chunk:
                attrs = found.get(wid)
                result[wid] = attrs
                if self.cache_size:
                    self.cache[wid] = attrs
                    while len(self.cache) > self.cache_size:
                        self.cache.popitem(last=False)
        return result

    def close(self):
        self.connection.close()


def run_self_checks():
    """Numerical examples, missingness, ties, repeated events, and disk lookup."""
    from math import isclose
    products = {
        1: (10, 100), 2: (20, 100), 3: (10, 200),
        4: (None, 200), 5: (30, None), 6: (0, 0),
    }
    d, b, c, stats = engineer_slate([1, 2, 3, 3, 999, None], [1, 2, 3, 999, None], products)
    assert d == 10
    assert all(isclose(x, y) for x, y in zip(b, [3/6, 1/6, 3/6, 0, 0]))
    assert all(isclose(x, y) for x, y in zip(c, [2/6, 2/6, 2/6, 0, 0]))
    assert stats["history_missing_product"] == 2
    assert engineer_slate([2, 1], [1], products)[0] == 10  # tie
    assert engineer_slate([1, 1, 2], [1], products)[1] == [2/3]
    assert engineer_slate([], [1], products)[:3] == (None, [0.0], [0.0])
    assert engineer_slate([999, None], [1], products)[:3] == (None, [0.0], [0.0])
    assert engineer_slate([4, 5], [4, 5], products)[:3] == (30, [0, 1/2], [1/2, 0])
    assert engineer_slate([6], [6, 6], products)[:3] == (0, [1.0, 1.0], [1.0, 1.0])
    assert engineer_slate([1], [], products)[:3] == (10, [], [])
    assert normalise_sequence(None) == []
    for invalid in (1.5, "1", True, 2**63):
        try:
            integer_id(invalid)
        except (TypeError, OverflowError):
            pass
        else:
            raise AssertionError(f"Invalid ID accepted: {invalid!r}")
    with tempfile.TemporaryDirectory() as tmp:
        db = Path(tmp) / "products.sqlite"
        con = sqlite3.connect(db)
        con.execute("CREATE TABLE products(wid INTEGER PRIMARY KEY, brand_id INTEGER, cate_id_4 INTEGER)")
        con.executemany("INSERT INTO products VALUES (?, ?, ?)", [(i, i % 7, i % 9) for i in range(1200)])
        con.commit()
        try:
            con.execute("INSERT INTO products VALUES (0, 99, 99)")
        except sqlite3.IntegrityError:
            con.rollback()
        else:
            raise AssertionError("Duplicate wid accepted")
        con.close()
        lookup = ProductLookup(db, cache_size=2)
        try:
            result = lookup.resolve([*range(1200), 5000, None])
            assert len(result) == 1201 and result[1199] == (1199 % 7, 1199 % 9)
            assert result[5000] is None and len(lookup.cache) <= 2
            assert lookup.resolve([1, 1, 5000]) == {1: (1, 1), 5000: None}
        finally:
            lookup.close()
    print("Self-checks passed: ratios, ties, repeated interactions, missingness, IDs, and bounded SQLite lookup.")


In [3]:
run_self_checks()

Self-checks passed: ratios, ties, repeated interactions, missingness, IDs, and bounded SQLite lookup.


## 3. Inspect the configured inputs

This only reads Parquet metadata and a few rows. It does not load a whole
split or the 12-million-product metadata table into memory.


In [4]:
try:
    import pyarrow as pa
    import pyarrow.parquet as pq
except ImportError as error:
    raise ImportError("Install requirements.txt in this notebook's Python environment first.") from error

print("PyArrow version:", pa.__version__)
for split, path in INPUT_PATHS.items():
    if not path.is_file():
        print(f"{split}: MISSING — {path}")
        continue
    pf = pq.ParquetFile(path)
    print(f"\n{split}: {pf.metadata.num_rows:,} rows, {len(pf.schema_arrow.names)} columns")
    for name in (HISTORY_COLUMN, CANDIDATE_COLUMN, "candidate_label_list"):
        if name in pf.schema_arrow.names:
            print(" ", pf.schema_arrow.field(name))
print("\nMetadata path:", PRODUCT_METADATA_PATH, "— exists:", PRODUCT_METADATA_PATH.exists())


PyArrow version: 25.0.1

train: 139,882 rows, 19 columns
  pyarrow.Field<history_wid_list: list<element: int64>>
  pyarrow.Field<candidate_wid_list: list<element: int64>>
  pyarrow.Field<candidate_label_list: list<element: int64>>

val: 16,924 rows, 19 columns
  pyarrow.Field<history_wid_list: list<element: int64>>
  pyarrow.Field<candidate_wid_list: list<element: int64>>
  pyarrow.Field<candidate_label_list: list<element: int64>>

test: 17,025 rows, 19 columns
  pyarrow.Field<history_wid_list: list<element: int64>>
  pyarrow.Field<candidate_wid_list: list<element: int64>>
  pyarrow.Field<candidate_label_list: list<element: int64>>

Metadata path: /Users/domi_bryan/Library/CloudStorage/GoogleDrive-domi@comp.nus.edu.sg/.shortcut-targets-by-id/1-dwM_44UXnFberNRNLzd2YLPRzRuRUUw/BT4222 Project Deliverables/data/JDSearch/product_metadata_clean.parquet — exists: True


## 4. Build or reuse a disk-backed product lookup

Only `wid`, `brand_id`, and `cate_id_4` are read. A SQLite primary key
prevents ambiguous product mappings. Metadata is streamed in batches, and
queries fetch only the distinct product IDs needed for the current behavior
batch. Repeated history events are counted later, after lookup.

The index fingerprint includes paths, sizes, modification times, selected
columns, and configured missing-ID rules. Set `REBUILD_METADATA_INDEX=True`
after a metadata change that preserves those file attributes. Reserve disk
space for the index and outputs; their sizes depend on the actual inputs.


In [5]:
def metadata_files(path):
    path = Path(path).expanduser().resolve()
    if path.is_file():
        return [path]
    if path.is_dir():
        files = sorted(path.rglob("*.parquet"))
        if files:
            return files
    raise FileNotFoundError(f"No product metadata Parquet files at {path}")


def metadata_fingerprint(files, columns, unknown_brand_ids, unknown_category_ids):
    payload = {
        "index_format": 1,
        "files": [{"path": str(p), "size": p.stat().st_size, "mtime_ns": p.stat().st_mtime_ns} for p in files],
        "columns": list(columns),
        "unknown_brand_ids": sorted(unknown_brand_ids),
        "unknown_category_ids": sorted(unknown_category_ids),
    }
    return hashlib.sha256(json.dumps(payload, sort_keys=True).encode()).hexdigest()


def build_product_index(metadata_path, index_path, *, columns=("wid", "brand_id", "cate_id_4"),
                        batch_size=100_000, unknown_brand_ids=frozenset(),
                        unknown_category_ids=frozenset(), rebuild=False):
    """Read only three metadata columns; enforce exactly one row per wid."""
    import pyarrow.parquet as pq
    files = metadata_files(metadata_path)
    columns = tuple(columns)
    if len(columns) != 3 or len(set(columns)) != 3:
        raise ValueError("Metadata columns must be three distinct names: wid, brand, cate_id_4.")
    unknown_brand_ids = {integer_id(v, allow_none=False) for v in unknown_brand_ids}
    unknown_category_ids = {integer_id(v, allow_none=False) for v in unknown_category_ids}
    fingerprint = metadata_fingerprint(files, columns, unknown_brand_ids, unknown_category_ids)
    index_path = Path(index_path).expanduser().resolve()
    if index_path in files:
        raise ValueError("The SQLite index path must not be a metadata input.")
    index_path.parent.mkdir(parents=True, exist_ok=True)
    if index_path.exists() and not rebuild:
        con = sqlite3.connect("file:" + quote(str(index_path)) + "?mode=ro", uri=True)
        try:
            stored = dict(con.execute("SELECT key, value FROM index_info"))
            if stored.get("fingerprint") == fingerprint:
                info = json.loads(stored["summary"])
                print(f"Reusing product index: {info['metadata_rows']:,} products")
                return info
        except sqlite3.DatabaseError:
            pass
        finally:
            con.close()

    temporary = index_path.with_name(index_path.name + ".building-" + uuid.uuid4().hex)
    con = sqlite3.connect(temporary)
    count = missing_brand = missing_category = 0
    started = time.perf_counter()
    try:
        con.execute("PRAGMA journal_mode=OFF")  # only the temporary build file
        con.execute("PRAGMA synchronous=OFF")
        con.execute("PRAGMA cache_size=-65536")
        con.execute("CREATE TABLE products(wid INTEGER PRIMARY KEY, brand_id INTEGER, cate_id_4 INTEGER)")
        con.execute("CREATE TABLE index_info(key TEXT PRIMARY KEY, value TEXT NOT NULL)")
        for source in files:
            parquet = pq.ParquetFile(source)
            missing = set(columns) - set(parquet.schema_arrow.names)
            if missing:
                raise ValueError(f"Metadata file {source} is missing columns: {sorted(missing)}")
            for batch in parquet.iter_batches(batch_size=batch_size, columns=list(columns)):
                values = [batch.column(batch.schema.get_field_index(c)).to_pylist() for c in columns]
                rows = []
                for wid, brand, category in zip(*values):
                    wid = integer_id(wid, allow_none=False)
                    brand, category = integer_id(brand), integer_id(category)
                    if brand in unknown_brand_ids:
                        brand = None
                    if category in unknown_category_ids:
                        category = None
                    rows.append((wid, brand, category))
                    missing_brand += int(brand is None)
                    missing_category += int(category is None)
                try:
                    con.executemany("INSERT INTO products VALUES (?, ?, ?)", rows)
                except sqlite3.IntegrityError as error:
                    raise ValueError(
                        f"Duplicate wid in product metadata ({source}). Provide one authoritative row per wid; "
                        "duplicates are rejected, including identical duplicates."
                    ) from error
                count += len(rows)
                con.commit()
                if count % 1_000_000 < len(rows):
                    print(f"Indexed {count:,} products")
        if count == 0:
            raise ValueError("Product metadata is empty.")
        if metadata_fingerprint(files, columns, unknown_brand_ids, unknown_category_ids) != fingerprint:
            raise RuntimeError("Metadata changed during indexing. Retry with stable input files.")
        info = {
            "metadata_rows": count, "metadata_missing_brand": missing_brand,
            "metadata_missing_category": missing_category, "fingerprint": fingerprint,
            "index_seconds": round(time.perf_counter() - started, 3),
        }
        con.executemany("INSERT INTO index_info VALUES (?, ?)", [
            ("fingerprint", fingerprint), ("summary", json.dumps(info)),
        ])
        con.commit()
        con.close()
        con = None
        os.replace(temporary, index_path)
        print(f"Product index ready: {count:,} products")
        return info
    finally:
        if con is not None:
            con.close()
        temporary.unlink(missing_ok=True)


## 5. Stream features to Parquet

Each split is processed independently, in original row and candidate order.
Parallel candidate labels, availability flags, and declared sequence lengths
are validated. Test labels may be absent. A temporary output is validated
and moved into place after each split completes.

No labels, `has_purchase`, `stratum`, or other splits' interactions are used
to calculate these features. The metadata lookup cache shares only static
product attributes. This assumes supplied histories contain only events
available before their query; temporal provenance is not inferred here.


In [6]:
from pathlib import Path
import pyarrow.parquet as pq

def iter_batches_with_context(source_path, batch_size):
    source_path = Path(source_path)

    with pq.ParquetFile(
        source_path,
        memory_map=False,
        pre_buffer=False,
        buffer_size=0,
    ) as pf:

        for row_group in range(pf.num_row_groups):
            rows_delivered = 0

            print(
                f"Reading {source_path.name} | "
                f"row group {row_group}/{pf.num_row_groups - 1}",
                flush=True,
            )

            try:
                for batch in pf.iter_batches(
                    batch_size=batch_size,
                    row_groups=[row_group],
                    use_threads=False,
                ):
                    rows_delivered += batch.num_rows
                    yield batch

            except OSError as error:
                raise OSError(
                    f"\nFailed input: {source_path}\n"
                    f"Row group: {row_group}\n"
                    f"Rows delivered from this group: {rows_delivered:,}\n"
                    f"Original error: {error}"
                ) from error

In [7]:
def feature_output_schema(input_schema, layout):
    import pyarrow as pa
    if len(set(input_schema.names)) != len(input_schema.names):
        raise ValueError("Duplicate column names in behavior input.")
    collisions = set(FEATURE_NAMES) & set(input_schema.names)
    if collisions:
        raise ValueError(f"Input already contains engineered columns: {sorted(collisions)}")
    if layout == "slate":
        schema = input_schema
        schema = schema.append(pa.field("hist_dominant_brand", pa.int64()))
        schema = schema.append(pa.field("brand_affinity_ratio", pa.list_(pa.float32())))
        schema = schema.append(pa.field("cate_affinity_ratio", pa.list_(pa.float32())))
    elif layout == "candidate":
        fields = [pa.field("source_row_id", pa.int64())]
        if "slate_id" in input_schema.names:
            fields.append(input_schema.field("slate_id"))
        fields += [pa.field("candidate_position", pa.int32()), pa.field("candidate_wid", pa.int64())]
        if "candidate_label_list" in input_schema.names:
            fields.append(pa.field("candidate_label", input_schema.field("candidate_label_list").type.value_type))
        fields += [pa.field("hist_dominant_brand", pa.int64()),
                   pa.field("brand_affinity_ratio", pa.float32()), pa.field("cate_affinity_ratio", pa.float32())]
        schema = pa.schema(fields)
    else:
        raise ValueError("OUTPUT_LAYOUT must be 'slate' or 'candidate'.")
    metadata = dict(schema.metadata or {})
    metadata[b"bt4222_feature_engineering"] = json.dumps({
        "layout": layout, "denominator": "full supplied history length",
        "dominant_brand_tie": "smallest numeric brand ID", "empty_affinity": 0.0,
    }).encode()
    return schema.with_metadata(metadata)

def process_split(source_path, destination_path, lookup, *, layout="slate", batch_size=128,
                  history_column="history_wid_list", candidate_column="candidate_wid_list",
                  overwrite=False):
    import pyarrow as pa
    import pyarrow.parquet as pq
    source_path, destination_path = Path(source_path).resolve(), Path(destination_path).resolve()
    if source_path == destination_path:
        raise ValueError("Output must not overwrite its source input.")
    if batch_size < 1:
        raise ValueError("batch_size must be positive")
    if destination_path.exists() and not overwrite:
        raise FileExistsError(f"Output exists: {destination_path}; set OVERWRITE_OUTPUTS=True to replace it.")
    parquet = pq.ParquetFile(source_path)
    for column in (history_column, candidate_column):
        if column not in parquet.schema_arrow.names:
            raise ValueError(f"Missing {column!r} in {source_path}")
        dtype = parquet.schema_arrow.field(column).type
        if not (pa.types.is_list(dtype) or pa.types.is_large_list(dtype)):
            raise TypeError(f"{column} must be an Arrow list column; found {dtype}")
    if "candidate_label_list" in parquet.schema_arrow.names:
        dtype = parquet.schema_arrow.field("candidate_label_list").type
        if not (pa.types.is_list(dtype) or pa.types.is_large_list(dtype)):
            raise TypeError("candidate_label_list must be a list column when present")
    schema = feature_output_schema(parquet.schema_arrow, layout)
    destination_path.parent.mkdir(parents=True, exist_ok=True)
    temporary = destination_path.with_name(destination_path.name + ".part-" + uuid.uuid4().hex)
    totals = Counter()
    source_row_offset, started = 0, time.perf_counter()
    writer = None
    try:
        writer = pq.ParquetWriter(temporary, schema, compression="zstd")
        for batch_number, batch in enumerate(iter_batches_with_context(source_path, batch_size), start=1):
            table = pa.Table.from_batches([batch])
            history = [normalise_sequence(x) for x in table[history_column].to_pylist()]
            candidates = [normalise_sequence(x) for x in table[candidate_column].to_pylist()]
            # Recompute slate_len from the actual candidate list.
            if "slate_len" in table.column_names:
                column_index = table.schema.get_field_index("slate_len")
                column_field = table.schema.field("slate_len")

                table = table.set_column(
                    column_index,
                    column_field,
                    pa.array(
                        [len(candidate_list) for candidate_list in candidates],
                        type=column_field.type,
                    ),
                )
            # Validate existing parallel lists without using their contents as features.
            for column, sequences in (("candidate_label_list", candidates), ("candidate_has_metadata", candidates),
                                      ("history_has_metadata", history)):
                if column in table.column_names:
                    for row, (parallel, seq) in enumerate(zip(table[column].to_pylist(), sequences)):
                        if parallel is not None and len(parallel) != len(seq):
                            raise ValueError(f"{column} length mismatch at source row {source_row_offset + row}")
            for column, sequences in (("hist_len", history), ("slate_len", candidates)):
                if column in table.column_names:
                    for row, (declared, seq) in enumerate(zip(table[column].to_pylist(), sequences)):
                        if declared is not None and declared != len(seq):
                            raise ValueError(f"{column} mismatch at source row {source_row_offset + row}")
            ids = {wid for seq in history for wid in seq if wid is not None}
            ids.update(wid for seq in candidates for wid in seq if wid is not None)
            products = lookup.resolve(ids)
            dominant, brand_values, category_values = [], [], []
            for hist, candidate in zip(history, candidates):
                d, b, c, stats = engineer_slate(hist, candidate, products)
                dominant.append(d)
                brand_values.append(b)
                category_values.append(c)
                totals.update(stats)

            if layout == "slate":
                result = table.append_column("hist_dominant_brand", pa.array(dominant, type=pa.int64()))
                result = result.append_column("brand_affinity_ratio", pa.array(brand_values, type=pa.list_(pa.float32())))
                result = result.append_column("cate_affinity_ratio", pa.array(category_values, type=pa.list_(pa.float32())))
                result = result.replace_schema_metadata(schema.metadata)
            else:
                # Compact candidate table: do not duplicate long history arrays per candidate.
                values = {field.name: [] for field in schema}
                slate_ids = table["slate_id"].to_pylist() if "slate_id" in table.column_names else None
                labels = table["candidate_label_list"].to_pylist() if "candidate_label_list" in table.column_names else None
                for row, candidate in enumerate(candidates):
                    for position, wid in enumerate(candidate):
                        values["source_row_id"].append(source_row_offset + row)
                        if slate_ids is not None:
                            values["slate_id"].append(slate_ids[row])
                        values["candidate_position"].append(position)
                        values["candidate_wid"].append(wid)
                        if labels is not None:
                            values["candidate_label"].append(labels[row][position] if labels[row] is not None else None)
                        values["hist_dominant_brand"].append(dominant[row])
                        values["brand_affinity_ratio"].append(brand_values[row][position])
                        values["cate_affinity_ratio"].append(category_values[row][position])
                result = pa.Table.from_pydict(values, schema=schema)
            writer.write_table(result)
            totals["output_rows"] += result.num_rows
            source_row_offset += batch.num_rows
            if batch_number % 50 == 0:
                print(f"{source_path.name}: processed {source_row_offset:,} / {parquet.metadata.num_rows:,} slates")
        writer.close()
        writer = None
        expected = parquet.metadata.num_rows if layout == "slate" else totals["candidate_events"]
        with pq.ParquetFile(temporary) as written:
            if written.metadata.num_rows != expected or not written.schema_arrow.equals(schema, check_metadata=False):
                raise RuntimeError("Parquet output row count or schema verification failed.")
        if destination_path.exists() and not overwrite:
            raise FileExistsError(f"Output was created during processing: {destination_path}")
        os.replace(temporary, destination_path)
    finally:
        if writer is not None:
            writer.close()
        parquet.close()
        temporary.unlink(missing_ok=True)
    totals["input_rows"] = source_row_offset
    summary = dict(totals)
    for name in ("history_events", "candidate_events", "empty_histories", "histories_without_known_brand", "output_rows"):
        summary.setdefault(name, 0)
    summary.update({"input": str(source_path), "output": str(destination_path), "layout": layout,
                    "seconds": round(time.perf_counter() - started, 3)})
    print(f"Saved {destination_path.name}: {summary['input_rows']:,} slates, {summary['candidate_events']:,} candidates")
    return summary


def run_pipeline(input_paths, metadata_path, output_directory, *, layout="slate", batch_size=128,
                 metadata_batch_size=100_000, cache_size=200_000, rebuild_index=False,
                 overwrite=False, metadata_columns=("wid", "brand_id", "cate_id_4"),
                 unknown_brand_ids=frozenset(), unknown_category_ids=frozenset(),
                 history_column="history_wid_list", candidate_column="candidate_wid_list"):
    import pyarrow.parquet as pq
    if set(input_paths) != {"train", "val", "test"}:
        raise ValueError("input_paths must contain exactly train, val, and test")
    if layout not in {"slate", "candidate"}:
        raise ValueError("layout must be slate or candidate")
    if batch_size < 1 or metadata_batch_size < 1:
        raise ValueError("Batch sizes must be positive")
    missing = [str(Path(p)) for p in input_paths.values() if not Path(p).is_file()]
    if not Path(metadata_path).exists():
        missing.append(str(metadata_path))
    if missing:
        raise FileNotFoundError("Missing required inputs (no features were generated):\n  " + "\n  ".join(missing))
    output_directory = Path(output_directory).expanduser().resolve()
    outputs = {split: output_directory / f"{split}.parquet" for split in ("train", "val", "test")}
    protected = {Path(p).expanduser().resolve() for p in input_paths.values()} | set(metadata_files(metadata_path))
    index_path = output_directory / "product_lookup.sqlite"
    report_path = output_directory / "feature_engineering_report.json"
    if protected & (set(outputs.values()) | {index_path, report_path}):
        raise ValueError("An output/index/report path conflicts with an input. Choose a separate OUTPUT_DIR.")
    if not overwrite:
        existing = [str(p) for p in (*outputs.values(), report_path) if p.exists()]
        if existing:
            raise FileExistsError("Outputs already exist; set OVERWRITE_OUTPUTS=True:\n  " + "\n  ".join(existing))
    # Check all behavior schemas before the expensive 12-million-product index.
    for split, path in input_paths.items():
        schema = pq.ParquetFile(path).schema_arrow
        for column in (history_column, candidate_column):
            if column not in schema.names:
                raise ValueError(f"{split} is missing required column {column!r}")
        feature_output_schema(schema, layout)
    index_info = build_product_index(
        metadata_path, index_path, columns=metadata_columns, batch_size=metadata_batch_size,
        unknown_brand_ids=unknown_brand_ids, unknown_category_ids=unknown_category_ids,
        rebuild=rebuild_index,
    )
    lookup = ProductLookup(index_path, cache_size=cache_size)
    report = {"metadata": index_info, "splits": {}}
    try:
        for split in ("train", "val", "test"):
            report["splits"][split] = process_split(
                input_paths[split], outputs[split], lookup, layout=layout, batch_size=batch_size,
                history_column=history_column, candidate_column=candidate_column, overwrite=overwrite,
            )
    finally:
        lookup.close()
    report["rules"] = {
        "denominator": "full supplied history length; missing metadata remains in denominator",
        "dominant_brand_tie": "smallest numeric brand ID",
        "missing_dominant_brand": None, "missing_or_empty_affinity": 0.0,
        "unknown_brand_ids": sorted(unknown_brand_ids), "unknown_category_ids": sorted(unknown_category_ids),
        "history_column": history_column, "candidate_column": candidate_column,
        "metadata_columns": list(metadata_columns),
    }
    temporary = report_path.with_name(report_path.name + ".part-" + uuid.uuid4().hex)
    try:
        temporary.write_text(json.dumps(report, indent=2), encoding="utf-8")
        os.replace(temporary, report_path)
    finally:
        temporary.unlink(missing_ok=True)
    return report


## 6. Run a tiny end-to-end check

This uses temporary synthetic files, independent of your real inputs. It
checks all splits, both layouts, missing test labels, duplicate slate IDs,
metadata index reuse, and invalid-input cleanup. The files are removed when
the checks finish. This check was authored but could not be executed in the
preparation environment, which did not have PyArrow available.


In [8]:
def run_parquet_smoke_checks():
    """Run a small end-to-end check before processing the real datasets."""
    import pyarrow as pa
    import pyarrow.parquet as pq
    from math import isclose
    with tempfile.TemporaryDirectory() as temporary:
        root = Path(temporary)
        metadata_path = root / "metadata.parquet"
        pq.write_table(pa.table({
            "wid": pa.array([1, 2, 3, 4], type=pa.int64()),
            "brand_id": pa.array([10, 20, 10, None], type=pa.int64()),
            "cate_id_4": pa.array([100, 100, 200, 200], type=pa.int64()),
        }), metadata_path)
        source = pa.table({
            "slate_id": pa.array([7, 7, 8, 9], type=pa.int32()),  # repeated ID must not merge rows
            "candidate_wid_list": pa.array([[1, 2, 999], [3], [1], []], type=pa.list_(pa.int64())),
            "candidate_label_list": pa.array([[1, 0, 0], [0], [0], []], type=pa.list_(pa.int64())),
            "history_wid_list": pa.array([[1, 3, 3, 999], [2, 1], [], [4]], type=pa.list_(pa.int64())),
            "hist_len": [4, 2, 0, 1], "slate_len": [3, 1, 1, 0],
        })
        inputs = {split: root / f"in_{split}.parquet" for split in ("train", "val", "test")}
        for split, path in inputs.items():
            # A test split without labels must also work.
            pq.write_table(source.drop(["candidate_label_list"]) if split == "test" else source, path)
        for layout in ("slate", "candidate"):
            report = run_pipeline(inputs, metadata_path, root / layout, layout=layout, batch_size=2,
                                  metadata_batch_size=2, cache_size=2)
            result = pq.read_table(root / layout / "val.parquet")
            assert result.num_rows == (4 if layout == "slate" else 5)
            assert report["splits"]["val"]["candidate_events"] == 5
            if layout == "slate":
                assert result["hist_dominant_brand"].to_pylist() == [10, 10, None, None]
                assert result["brand_affinity_ratio"].to_pylist() == [[0.75, 0, 0], [0.5], [0], []]
                assert result["cate_affinity_ratio"].to_pylist() == [[0.25, 0.25, 0], [0], [0], []]
                for name in source.column_names:
                    assert result[name].equals(source[name]), f"Source column changed: {name}"
            else:
                assert result["source_row_id"].to_pylist() == [0, 0, 0, 1, 2]
                assert result["candidate_position"].to_pylist() == [0, 1, 2, 0, 0]
                assert result["candidate_wid"].to_pylist() == [1, 2, 999, 3, 1]
                assert result["candidate_label"].to_pylist() == [1, 0, 0, 0, 0]
                assert result["hist_dominant_brand"].to_pylist() == [10, 10, 10, 10, None]
                assert all(isclose(a, b) for a, b in zip(result["brand_affinity_ratio"].to_pylist(), [0.75, 0, 0, 0.5, 0]))
            assert "candidate_label" not in pq.read_schema(root / layout / "test.parquet").names
            # A second call must reuse an unchanged metadata index.
            info = build_product_index(metadata_path, root / layout / "product_lookup.sqlite")
            assert info["metadata_rows"] == 4

        duplicate_path = root / "duplicate_metadata.parquet"
        pq.write_table(pa.table({"wid": [1, 1], "brand_id": [10, 20], "cate_id_4": [100, 200]}), duplicate_path)
        try:
            build_product_index(duplicate_path, root / "bad.sqlite")
        except ValueError as error:
            assert "Duplicate wid" in str(error)
        else:
            raise AssertionError("Duplicate metadata was not rejected")
        assert not (root / "bad.sqlite").exists()

        lookup = ProductLookup(root / "slate" / "product_lookup.sqlite")
        try:
            bad = root / "bad_alignment.parquet"
            pq.write_table(pa.table({"history_wid_list": [[1]], "candidate_wid_list": [[1, 2]],
                                     "candidate_label_list": [[1]]}), bad)
            try:
                process_split(bad, root / "should_not_exist.parquet", lookup)
            except ValueError as error:
                assert "length mismatch" in str(error)
            else:
                raise AssertionError("Misaligned candidates and labels were not rejected")
            assert not (root / "should_not_exist.parquet").exists()
        finally:
            lookup.close()
    print("Parquet smoke checks passed: all three splits, both layouts, labels, row order, index reuse, and failure cleanup.")


run_parquet_smoke_checks()


Product index ready: 4 products
Reading in_train.parquet | row group 0/0
Saved train.parquet: 4 slates, 5 candidates
Reading in_val.parquet | row group 0/0
Saved val.parquet: 4 slates, 5 candidates
Reading in_test.parquet | row group 0/0
Saved test.parquet: 4 slates, 5 candidates
Reusing product index: 4 products
Product index ready: 4 products
Reading in_train.parquet | row group 0/0
Saved train.parquet: 4 slates, 5 candidates
Reading in_val.parquet | row group 0/0
Saved val.parquet: 4 slates, 5 candidates
Reading in_test.parquet | row group 0/0
Saved test.parquet: 4 slates, 5 candidates
Reusing product index: 4 products
Reading bad_alignment.parquet | row group 0/0
Parquet smoke checks passed: all three splits, both layouts, labels, row order, index reuse, and failure cleanup.


## 7. Generate the three real output files

All required paths and destination conflicts are checked before processing.
Missing files raise an error listing the inputs to supply. Original inputs
are protected. Existing outputs require `OVERWRITE_OUTPUTS=True`.

Choose `OUTPUT_LAYOUT="candidate"` in the configuration cell if you need
one scalar feature value per candidate row. Use a separate `OUTPUT_DIR`
when keeping both layouts.


In [9]:
report = run_pipeline(
    INPUT_PATHS,
    PRODUCT_METADATA_PATH,
    OUTPUT_DIR,
    layout=OUTPUT_LAYOUT,
    batch_size=BATCH_SIZE,
    metadata_batch_size=METADATA_BATCH_SIZE,
    cache_size=LOOKUP_CACHE_SIZE,
    rebuild_index=REBUILD_METADATA_INDEX,
    overwrite=OVERWRITE_OUTPUTS,
    metadata_columns=METADATA_COLUMNS,
    unknown_brand_ids=UNKNOWN_BRAND_IDS,
    unknown_category_ids=UNKNOWN_CATEGORY_IDS,
    history_column=HISTORY_COLUMN,
    candidate_column=CANDIDATE_COLUMN,
)
print("\nAll three splits completed:")
for split, summary in report["splits"].items():
    print(f"{split}: {summary['input_rows']:,} source slates -> {summary['output_rows']:,} output rows")


Indexed 1,000,000 products
Indexed 2,000,000 products
Indexed 3,000,000 products
Indexed 4,000,000 products
Indexed 5,000,000 products
Indexed 6,000,000 products
Indexed 7,000,000 products
Indexed 8,000,000 products
Indexed 9,000,000 products
Indexed 10,000,000 products
Indexed 11,000,000 products
Indexed 12,000,000 products
Product index ready: 12,141,247 products
Reading train_sampled.parquet | row group 0/0
train_sampled.parquet: processed 6,400 / 139,882 slates
train_sampled.parquet: processed 12,800 / 139,882 slates
train_sampled.parquet: processed 19,200 / 139,882 slates
train_sampled.parquet: processed 25,600 / 139,882 slates
train_sampled.parquet: processed 32,000 / 139,882 slates
train_sampled.parquet: processed 38,400 / 139,882 slates
train_sampled.parquet: processed 44,800 / 139,882 slates
train_sampled.parquet: processed 51,200 / 139,882 slates
train_sampled.parquet: processed 57,600 / 139,882 slates
train_sampled.parquet: processed 64,000 / 139,882 slates
train_sampled.par

## 8. Inspect outputs and missingness

The JSON report records input/output paths, row and candidate counts,
missing product/brand/category interaction counts, empty histories, and
feature rules. Counts are interaction-weighted, not unique-product counts.
Preview only a small batch below.


In [11]:
for split in ("train", "val", "test"):
    path = OUTPUT_DIR / f"{split}.parquet"
    pf = pq.ParquetFile(path)
    print(f"\n{path}: {pf.metadata.num_rows:,} rows")
    columns = (["slate_id"] if "slate_id" in pf.schema_arrow.names else []) + list(FEATURE_NAMES)
    batch = next(pf.iter_batches(batch_size=3, columns=columns), None)
    if batch is not None:
        for row in batch.to_pylist():
            print({name: (value[:6] if isinstance(value, list) else value) for name, value in row.items()})
    summary = report["splits"][split]
    for field in ("history_missing_product", "history_missing_brand", "history_missing_category",
                  "candidate_missing_product", "candidate_missing_brand", "candidate_missing_category"):
        denominator = summary["history_events" if field.startswith("history_") else "candidate_events"]
        count = summary.get(field, 0)
        print(f"  {field}: {count:,}/{denominator:,} ({count / denominator:.2%})" if denominator else f"  {field}: 0/0")
print("\nAudit:", OUTPUT_DIR / "feature_engineering_report.json")



/Users/domi_bryan/Library/CloudStorage/GoogleDrive-domi@comp.nus.edu.sg/.shortcut-targets-by-id/1-dwM_44UXnFberNRNLzd2YLPRzRuRUUw/BT4222 Project Deliverables/data/JDSearch/model2_inputs/Dom/train.parquet: 139,882 rows
{'slate_id': 0, 'hist_dominant_brand': 47369831, 'brand_affinity_ratio': [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], 'cate_affinity_ratio': [0.0, 0.0, 0.0, 0.0, 0.0, 0.0]}
{'slate_id': 1, 'hist_dominant_brand': 22667925, 'brand_affinity_ratio': [0.0, 0.0, 0.0, 0.0, 0.0, 0.0], 'cate_affinity_ratio': [0.0, 0.0, 0.0, 0.0, 0.0, 0.0]}
{'slate_id': 3, 'hist_dominant_brand': 87675207, 'brand_affinity_ratio': [0.039024390280246735, 0.039024390280246735, 0.039024390280246735, 0.039024390280246735, 0.039024390280246735, 0.039024390280246735], 'cate_affinity_ratio': [0.039024390280246735, 0.0, 0.039024390280246735, 0.039024390280246735, 0.039024390280246735, 0.0]}
  history_missing_product: 1,039,665/21,401,603 (4.86%)
  history_missing_brand: 1,039,665/21,401,603 (4.86%)
  history_missing_cat